# Waste YOLO Training — Run All

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**

Trains both models and downloads `garbage_best.pt` + `plastic_best.pt` at the end.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SMART CAMPUS WASTE DETECTION — FULL AUTO TRAINING
#  Just Run All. No settings to change.
# ══════════════════════════════════════════════════════════════

!pip install -q ultralytics

import shutil, zipfile, torch
from pathlib import Path
from ultralytics import YOLO

CONTENT = Path('/content')
EPOCHS, BATCH, IMG_SIZE = 50, 16, 640
BASE_MODEL = 'yolo11n.pt'
OUT = CONTENT / 'trained_models'
OUT.mkdir(exist_ok=True)

print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — enable T4 in Runtime settings')

# ── Auto-find or extract datasets (zero config) ──
def find_root():
    candidates = [CONTENT, CONTENT / 'Waste_YOLO_Colab_Ready', Path.cwd()]
    for c in candidates:
        if (c / 'GARBAGE CLASSIFICATION' / 'train' / 'images').exists():
            return c
    for z in sorted(CONTENT.glob('*.zip')):
        print(f'Extracting {z.name}...')
        with zipfile.ZipFile(z, 'r') as zf:
            zf.extractall(CONTENT)
        for c in candidates:
            if (c / 'GARBAGE CLASSIFICATION' / 'train' / 'images').exists():
                return c
        for sub in CONTENT.iterdir():
            if sub.is_dir() and (sub / 'GARBAGE CLASSIFICATION').exists():
                return sub
    # Fallback: pop-up to pick zip from your PC
    from google.colab import files
    print('Select Waste_YOLO_Colab_Ready.zip from your PC...')
    up = files.upload()
    for fname, data in up.items():
        zp = CONTENT / fname
        zp.write_bytes(data)
        with zipfile.ZipFile(zp, 'r') as zf:
            zf.extractall(CONTENT)
    for c in candidates:
        if (c / 'GARBAGE CLASSIFICATION' / 'train' / 'images').exists():
            return c
    for sub in CONTENT.iterdir():
        if sub.is_dir() and (sub / 'GARBAGE CLASSIFICATION').exists():
            return sub
    raise FileNotFoundError('Could not find datasets. Run CREATE_PACKAGE.bat on your PC first.')

ROOT = find_root()
GARBAGE_DIR = ROOT / 'GARBAGE CLASSIFICATION'
PLASTIC_DIR = ROOT / 'Plastic_trained'
print(f'Root: {ROOT}')
print(f'Garbage images: {len(list((GARBAGE_DIR / "train" / "images").glob("*")))}')
print(f'Plastic images:  {len(list((PLASTIC_DIR / "train" / "images").glob("*")))}')

def write_yaml(folder, names):
    p = folder / 'data.yaml'
    p.write_text(f'path: {folder}\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: {len(names)}\nnames: {names}\n')
    return p

garbage_yaml = write_yaml(GARBAGE_DIR, ['BIODEGRADABLE','CARDBOARD','GLASS','METAL','PAPER','PLASTIC'])
plastic_yaml = write_yaml(PLASTIC_DIR, ['Bag','Bottle','Cup'])

def train_and_save(yaml_path, run_name, out_name):
    print(f'\n{"="*50}\nTraining {run_name}...\n{"="*50}')
    model = YOLO(BASE_MODEL)
    r = model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
                    project=str(OUT/'runs'), name=run_name, exist_ok=True, patience=15, device=0)
    best = Path(r.save_dir) / 'weights' / 'best.pt'
    dest = OUT / out_name
    shutil.copy2(best, dest)
    print(f'Saved: {dest}')
    return dest

garbage_pt = train_and_save(garbage_yaml, 'garbage_train', 'garbage_best.pt')
plastic_pt = train_and_save(plastic_yaml, 'plastic_train', 'plastic_best.pt')

print('\n' + '='*50)
print('DONE! Downloading models to your PC...')
print('='*50)
print('Copy downloaded files to:')
print('  garbage_best.pt -> ml_service/models/garbage_best.pt')
print('  plastic_best.pt -> ml_service/models/plastic_best.pt')

from google.colab import files
files.download(str(garbage_pt))
files.download(str(plastic_pt))
print('\nAll finished!')